# Live monitor

**Goal:** run the whole system as a service: notice newly published nights, search them with all three searches, and raise
alerts for unknown objects. Read `docs/live_monitor.md` alongside this notebook.

Here we use **replay mode**: the monitor is pointed at a past window, so a night from 2019 is processed exactly as if it had
just appeared in the archive. That's how a live pipeline is tested before it's trusted.

In [ ]:
import json
import shutil
import numpy as np
import pandas as pd
import torch

from asthunt.config.load_config import load_config
from asthunt.acquire.load_unit_metadata import load_unit_metadata
from asthunt.models.load_model_bundle import load_model_bundle
from asthunt.pipeline.run_live_monitor import run_live_monitor

project = load_config("project")
configs = {name: load_config(name) for name in ["acquire", "inject", "detect", "link", "fast", "deep", "live"]}
device = "mps" if torch.backends.mps.is_available() else "cpu"
bundle = load_model_bundle(project["paths"]["models"], configs["live"]["bundle"])
print("watching:", configs["live"]["watch"], "| bundle:", configs["live"]["bundle"])

## 1. A replay window and a separate output folder
We replay one hunt night (2019-01-23) and write state and alerts into `reports/replay/`, so the replay never mixes with real
monitoring. The archive query uses the cached table (the same columns the live query returns).

In [ ]:
replay_paths = {**project["paths"], "reports": project["paths"]["reports"] / "replay"}
shutil.rmtree(replay_paths["reports"], ignore_errors=True)          # a clean replay every time
configs["live"] = {**configs["live"], "start_after_night": "2019-01-22", "stop_before_night": "2019-01-24"}


def cached_query(field, ccd, quadrant):
    return load_unit_metadata({"field": field, "ccd": ccd, "quadrant": quadrant}, project["paths"]["raw"])

## 2. One monitoring pass

In [ ]:
searched = run_live_monitor(configs, replay_paths, bundle, device=device, query_metadata=cached_query)
pd.DataFrame(searched)

In [ ]:
alerts_file = replay_paths["reports"] / "alerts.jsonl"
alerts = [json.loads(line) for line in alerts_file.read_text().splitlines()] if alerts_file.exists() else []
print(f"{len(alerts)} alert(s)")
pd.DataFrame(alerts)

## 3. A second pass finds nothing new (the state file remembers)

In [ ]:
print(run_live_monitor(configs, replay_paths, bundle, device=device, query_metadata=cached_query))
print(json.loads((replay_paths["reports"] / "live_state.json").read_text()))

## 4. Running it for real
```bash
python -m asthunt.cli watch            # every poll_minutes (configs/live.yaml), from start_after_night on
python -m asthunt.cli watch --once     # one pass, e.g. from cron or launchd
```
New nights appear in the public archive with a delay, so "live" means *as soon as data is public*. For true real-time,
`run_full_search` would be fed by a telescope or ZTF's alert stream instead of the archive query.

## Exercises
1. Change the replay window to include 2019-01-20 too. Which searches run on a 3-frame night?
2. What happens to `alerts.jsonl` if you delete `live_state.json` and run again? Why is that dangerous in production?
3. Where would you send alerts (e-mail, Slack)? Which single function would you change?